# 1. Environment preparation

`DataLoader`:https://docs.pytorch.org/docs/main/utils/data.html#torch.utils.data.DataLoader

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
import random
import urllib.request
import zipfile
import os

from torchvision import datasets, transforms, models
from torchvision.models import resnet18, resnet34, resnet50, resnet101, resnet152
from pathlib import Path
from PIL import Image
from torchvision.models import ResNet18_Weights
from torch.utils.data import DataLoader

# 2. Set up device and random seed

In [ ]:
SEED = 42
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

# 3. Prepare for the data set

## (1) Download the dataset

In [ ]:
url = "https://zenodo.org/records/5226945/files/cats_dogs_light.zip?download=1"
zip_path = Path('/content/cats_dogs_light.zip')

# Download only if the file is not
if not zip_path.exists():
  urllib.request.urlretrieve(url, zip_path)

print("Dataset is downloaded")

Dataset is downloaded


## (2) Read the images and convert them into tensors  
python formal tech doc: https://docs.python.org/3/library/zipfile.html#zipfile.ZipFile.namelist


In [ ]:
preprocess = ResNet18_Weights.IMAGENET1K_V1.transforms()    # Image Processing Pipeline

dataset = []

with zipfile.ZipFile(zip_path) as archive:

  for path in archive.namelist():

    if "/test/" not in path.lower():
      continue

    filename = Path(path).name.lower()

    if not filename.endswith((".jpg", ".jpeg", ".png")):
      continue

    if filename.startswith("cat."):
      label = 0
    elif filename.startswith("dog."):
      label = 1
    else:
      continue

    with archive.open(path) as file:
      image = Image.open(file).convert("RGB")

    image = preprocess(image)

    dataset.append((image, label))

print("Number of images:", len(dataset))
print("Shape of one image:", dataset[0][0].shape)



Number of images: 400
Shape of one image: torch.Size([3, 224, 224])


## (3) Use a `DataLoader` to get batches

In [ ]:
loader = DataLoader(
    dataset,
    batch_size = 4,
    shuffle = False
)

for images, labels in loader:
  print("Image batch shape:", image.shape)
  print("Label batch shape:", labels.shape)
  print("Labels:", labels)
  break

Image batch shape: torch.Size([3, 224, 224])
Label batch shape: torch.Size([4])
Labels: tensor([1, 1, 1, 1])


# 4. Set up model

(1) Test the forward propagation pipeline

In [ ]:
model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)
model = model.to(device)
model.eval()

features = []

with.torch.inference_model():

  for images, labels in loader:

    image = image.t0(device)

    # The beginning of ResNet
    x = model.conv1(images)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpoool(x)

  # Visit each complete residual block
  for stage in [model.layer1, model.layer2, model.layer3, model.layer4]:
    for block in stage:
      x = block(x)

      feature = x.mean(dim=(2, 3))

      features.append(feature.cpu())

  # First batch only for this initial check
  break

print('Number of saved block outputs:', len(features))

for feature in features:
  print(feature.shape)

(2) Extract features form the entire dataset

In [ ]:
features = [[] for _ in range(8)]
all_labels = []

model.eval()

with torch.inference_mode():

  for images, labels in loader:

    images = images.to(device)

    # Save labels for later cat/dog separation
    all_labels.append(labels)

    # Beginning of ResNet
    x = model.conv1(images)
    x = model.bn1(x)
    x = model.relu(x)
    x = model.maxpool(x)


    for layer_idx, stage in enumerate([model.layer1, model.layer2, model.layer3, model.layer4], 1):


      for block_idx, block in enumerate(stage, 1):

        x = block(x)

        # Global Average Pooling
        feature = x.mean(dim=(2,3))

        # L2 normalization
        feature = torch.nn.functional.normalize(
            feature, p=2, dim=1
        )

        # Store features for this block
        flat_index = (layer_idx - 1) * 2 + (block_idx - 1)
        features[flat_index].append(feature.cpu())

        chapter_name = f"Processing block{layer_idx}.{block_idx}, saved to index {flat_index}"
        print(chapter_name)









The data structure of `all_labels`
```
all_labels = [
    tensor([1, 1, 1, 1]),   # four labels in the first batch
    tensor([1, 1, 1, 1]),   # The second batch
    ...
    tensor([0, 0, 0, 0]),   # Another batch
    ...
]                            # We have 100 elements in total
```  
The data structure of `features`
```
features = [
    list_0,   # Output features from block 1 (layer1.1)
    list_1,   # Output features from block 2 (layer1.2)
    list_2,   # Output features from block 3 (layer2.1)
    list_3,   # Output features from block 4 (layer2.2)
    list_4,   # Output features from block 5 (layer3.1)
    list_5,   # Output features from block 6 (layer3.2)
    list_6,   # Output features from block 7 (layer4.1)
    list_7    # Output features from block 8 (layer4.2)
]
```  
The data structure of each `list_i`(take `list_0` as an example)
```
list_0 = [
    tensor_batch_1,   # shape: torch.Size([4, 64])
    tensor_batch_2,   # shape: torch.Size([4, 64])
    tensor_batch_3,   # shape: torch.Size([4, 64])
    ...               # (94 tensors omitted)
    tensor_batch_100  # shape: torch.Size([4, 64])
]
```

# 5. Randomly sample 100 cats and 100 dogs

In [ ]:
# Get indices for each class
all_labels = torch.cat(all_labels)
cat_indices = torch.where(all_labels == 0)[0]
dog_indices = torch.where(all_labels == 1)[0]

# Set a reproducible sampling seed
generator = torch.Generator().manual_seed(20261015)

# Randomly select 100 images form each class
cat_sample = cat_indices[
    torch.randperm(len(cat_indices),generator=generator)[:100]
]

dog_sample = dog_indices[
    torch.randperm(len(dog_indices),generator=generator)[:100]
]

print(len(cat_sample))
print(len(dog_sample))

Here we touch a very important concept: Global Random Seed and Local Random Seed:

1. Global Random Seed:

```
SEED = 42
def set_seed(seed):
  random.seed(seed)
  np.random.seed(seed)
  torch.manual_seed(seed)
  if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
set_seed(SEED)
```

This random seed governs the following things:

- model initialization(you can't set random seed for every block/layer)
- Dropout and data augmentation
- Third-party library(numpy/random/PIL)

2. Local Random Seed:

- Dataset Splitting and Sampling
- Generate fixed, reproducible random noise / masks
- Multiple independent runs




# 6.Calculate Participation Ratio and do the visualization

(1) Define the function to calculate the participation ratio

In [ ]:
def calculate_pr(x):

  # Use higher precision
  x = x.double()

  # Center features
  x = x - x.mean(dim=0, keepdim=True)

  # Gram matrix
  gram = x @ x.T

  # Eigenvalues
  eigenvalues = torch.linalg.eigvalsh(gram)
  eigenvalues = eigenvalues.clamp_min(0)

  # Participation Ratio
  numerator = eigenvalues.sum() ** 2
  denominator = (eigenvalues ** 2).sum()

  if denominator == 0:
    return float("nan")

  return (numerator / denominator).item()


The data structure of `eigenvalues`
```
tensor([0.0000e+00, 0.0000e+00, 0.0000e+00,  ..., 1.2345e-02, 8.5678e-01, 3.1415e+01],
       dtype=torch.float64)
```  
This is just an example rather than the real output.

(2) Calculate PR across all eight residual blocks

In [ ]:
cat_pr = []
dog_pr = []

for block_features in features:

  cat_features = block_features[cat_sample]
  dog_features = block_features[dog_sample]

  cat_pr.append(calculate_pr(cat_features))
  dog_pr.append(calculate_pr(dog_features))

print('Cat PR:',cat_pr)
print('Dog PR:',dog_pr)

The data structure of `cat_pr`/`dog_pr`  
```
cat_pr = [PR_block1, PR_block2, PR_block3, PR_block4, PR_block5, PR_block6, PR_block7, PR_block8]
```

(3) Visualize the PR curves

In [ ]:
blocks = ['1.1', '1.2', '2.1', '2.2', '3.1', '3.2', '4.1', '4.2']

plt.figure(figsize=(9, 5))

plt.plot(blocks, cat_pr, marker="o", label="Cats")
plt.plot(blocks, dog_pr, marker="o", label="Dogs")

plt.xlabel("Residual Block")
plt.ylabel("Participation Ratio")
plt.title("Within-Class PR Across ResNet-18")
plt.xticks(blocks)
plt.legend()
plt.grid(alpha=0.3)

plt.show()
